# Multi-Agent System

This notebook demonstrates an agentic workflow with two specialized agents:
- **Math Agent**: Handles mathematical operations
- **String Agent**: Handles string manipulations

Each agent has specific capabilities and can be orchestrated to work together in complex workflows.

## 1. Agent Base Class

First, let's define a base Agent class that all specialized agents will inherit from.

In [ ]:
from typing import Any, Dict, List
from abc import ABC, abstractmethod
import json

class Agent(ABC):
    """Base class for all agents in the multi-agent system."""
    
    def __init__(self, name: str):
        self.name = name
        self.history = []
    
    @abstractmethod
    def get_capabilities(self) -> List[str]:
        """Return a list of capabilities this agent has."""
        pass
    
    @abstractmethod
    def execute(self, task: str, **kwargs) -> Any:
        """Execute a task and return the result."""
        pass
    
    def log_action(self, task: str, result: Any):
        """Log an action to the agent's history."""
        self.history.append({
            'task': task,
            'result': result,
            'agent': self.name
        })
    
    def get_history(self) -> List[Dict]:
        """Return the agent's action history."""
        return self.history
    
    def __str__(self):
        return f"Agent({self.name})"

## 2. Math Agent

The Math Agent handles various mathematical operations including basic arithmetic, statistics, and algebraic functions.

In [ ]:
import math
import statistics
from typing import List, Union

class MathAgent(Agent):
    """Agent specialized in mathematical operations."""
    
    def __init__(self):
        super().__init__("MathAgent")
    
    def get_capabilities(self) -> List[str]:
        return [
            'add', 'subtract', 'multiply', 'divide',
            'power', 'sqrt', 'factorial',
            'mean', 'median', 'std_dev',
            'sin', 'cos', 'tan'
        ]
    
    def execute(self, task: str, **kwargs) -> Any:
        """Execute a mathematical task."""
        result = None
        
        try:
            if task == 'add':
                result = kwargs['a'] + kwargs['b']
            elif task == 'subtract':
                result = kwargs['a'] - kwargs['b']
            elif task == 'multiply':
                result = kwargs['a'] * kwargs['b']
            elif task == 'divide':
                if kwargs['b'] == 0:
                    raise ValueError("Division by zero")
                result = kwargs['a'] / kwargs['b']
            elif task == 'power':
                result = kwargs['base'] ** kwargs['exponent']
            elif task == 'sqrt':
                result = math.sqrt(kwargs['value'])
            elif task == 'factorial':
                result = math.factorial(kwargs['n'])
            elif task == 'mean':
                result = statistics.mean(kwargs['numbers'])
            elif task == 'median':
                result = statistics.median(kwargs['numbers'])
            elif task == 'std_dev':
                result = statistics.stdev(kwargs['numbers'])
            elif task == 'sin':
                result = math.sin(kwargs['angle'])
            elif task == 'cos':
                result = math.cos(kwargs['angle'])
            elif task == 'tan':
                result = math.tan(kwargs['angle'])
            else:
                raise ValueError(f"Unknown task: {task}")
            
            self.log_action(task, result)
            return result
        
        except Exception as e:
            error_msg = f"Error executing {task}: {str(e)}"
            self.log_action(task, error_msg)
            return error_msg

## 3. String Agent

The String Agent handles various string manipulation operations including transformations, analysis, and formatting.

In [ ]:
import re
from typing import List

class StringAgent(Agent):
    """Agent specialized in string manipulation operations."""
    
    def __init__(self):
        super().__init__("StringAgent")
    
    def get_capabilities(self) -> List[str]:
        return [
            'uppercase', 'lowercase', 'capitalize', 'title_case',
            'reverse', 'count_chars', 'count_words',
            'remove_whitespace', 'replace', 'extract_numbers',
            'is_palindrome', 'concat', 'split'
        ]
    
    def execute(self, task: str, **kwargs) -> Any:
        """Execute a string manipulation task."""
        result = None
        
        try:
            text = kwargs.get('text', '')
            
            if task == 'uppercase':
                result = text.upper()
            elif task == 'lowercase':
                result = text.lower()
            elif task == 'capitalize':
                result = text.capitalize()
            elif task == 'title_case':
                result = text.title()
            elif task == 'reverse':
                result = text[::-1]
            elif task == 'count_chars':
                result = len(text)
            elif task == 'count_words':
                result = len(text.split())
            elif task == 'remove_whitespace':
                result = ''.join(text.split())
            elif task == 'replace':
                result = text.replace(kwargs['old'], kwargs['new'])
            elif task == 'extract_numbers':
                result = [int(num) for num in re.findall(r'\d+', text)]
            elif task == 'is_palindrome':
                cleaned = re.sub(r'[^a-zA-Z0-9]', '', text.lower())
                result = cleaned == cleaned[::-1]
            elif task == 'concat':
                result = kwargs['string1'] + kwargs['string2']
            elif task == 'split':
                delimiter = kwargs.get('delimiter', ' ')
                result = text.split(delimiter)
            else:
                raise ValueError(f"Unknown task: {task}")
            
            self.log_action(task, result)
            return result
        
        except Exception as e:
            error_msg = f"Error executing {task}: {str(e)}"
            self.log_action(task, error_msg)
            return error_msg

## 4. Agent Orchestrator

The Orchestrator manages multiple agents and routes tasks to the appropriate agent based on capabilities.

In [ ]:
class AgentOrchestrator:
    """Orchestrates multiple agents and routes tasks appropriately."""
    
    def __init__(self):
        self.agents = {}
    
    def register_agent(self, agent: Agent):
        """Register an agent with the orchestrator."""
        self.agents[agent.name] = agent
        print(f"Registered {agent.name} with capabilities: {agent.get_capabilities()}")
    
    def find_agent(self, task: str) -> Agent:
        """Find an agent capable of handling the given task."""
        for agent in self.agents.values():
            if task in agent.get_capabilities():
                return agent
        return None
    
    def execute_task(self, task: str, **kwargs) -> Any:
        """Execute a task by routing it to the appropriate agent."""
        agent = self.find_agent(task)
        
        if agent is None:
            return f"No agent found capable of handling task: {task}"
        
        print(f"Routing task '{task}' to {agent.name}")
        result = agent.execute(task, **kwargs)
        return result
    
    def execute_workflow(self, workflow: List[Dict]) -> List[Any]:
        """Execute a multi-step workflow across multiple agents."""
        results = []
        
        for step in workflow:
            task = step['task']
            params = step.get('params', {})
            
            # Allow referencing previous results
            if 'use_previous_result' in params:
                step_index = params['use_previous_result']
                if step_index < len(results):
                    params[params.get('result_key', 'value')] = results[step_index]
            
            result = self.execute_task(task, **params)
            results.append(result)
        
        return results
    
    def get_all_capabilities(self) -> Dict[str, List[str]]:
        """Get capabilities of all registered agents."""
        return {name: agent.get_capabilities() for name, agent in self.agents.items()}
    
    def get_agent_history(self, agent_name: str) -> List[Dict]:
        """Get the action history of a specific agent."""
        if agent_name in self.agents:
            return self.agents[agent_name].get_history()
        return []

## 5. Initialize the Multi-Agent System

Now let's create our agents and register them with the orchestrator.

In [ ]:
# Create the orchestrator
orchestrator = AgentOrchestrator()

# Create and register agents
math_agent = MathAgent()
string_agent = StringAgent()

orchestrator.register_agent(math_agent)
orchestrator.register_agent(string_agent)

print("\nMulti-Agent System initialized successfully!")

## 6. Example: Math Agent Operations

In [ ]:
print("=== Math Agent Examples ===")

# Basic arithmetic
result1 = orchestrator.execute_task('add', a=15, b=27)
print(f"Addition: 15 + 27 = {result1}")

result2 = orchestrator.execute_task('multiply', a=8, b=9)
print(f"Multiplication: 8 * 9 = {result2}")

# Advanced operations
result3 = orchestrator.execute_task('power', base=2, exponent=10)
print(f"Power: 2^10 = {result3}")

result4 = orchestrator.execute_task('sqrt', value=144)
print(f"Square root: √144 = {result4}")

# Statistics
numbers = [10, 20, 30, 40, 50]
result5 = orchestrator.execute_task('mean', numbers=numbers)
print(f"Mean of {numbers} = {result5}")

result6 = orchestrator.execute_task('median', numbers=numbers)
print(f"Median of {numbers} = {result6}")

## 7. Example: String Agent Operations

In [ ]:
print("=== String Agent Examples ===")

# Case transformations
text = "hello world from multi-agent system"
result1 = orchestrator.execute_task('uppercase', text=text)
print(f"Uppercase: {result1}")

result2 = orchestrator.execute_task('title_case', text=text)
print(f"Title case: {result2}")

# String analysis
result3 = orchestrator.execute_task('count_words', text=text)
print(f"Word count: {result3}")

result4 = orchestrator.execute_task('count_chars', text=text)
print(f"Character count: {result4}")

# String manipulation
result5 = orchestrator.execute_task('reverse', text="Python")
print(f"Reverse: {result5}")

# Extract numbers from text
text_with_numbers = "I have 3 apples and 5 oranges, total 8 fruits"
result6 = orchestrator.execute_task('extract_numbers', text=text_with_numbers)
print(f"Numbers in text: {result6}")

# Palindrome check
result7 = orchestrator.execute_task('is_palindrome', text="A man a plan a canal Panama")
print(f"Is palindrome: {result7}")

## 8. Example: Complex Workflow

This example demonstrates how multiple agents can work together in a coordinated workflow.

In [ ]:
print("=== Complex Multi-Agent Workflow ===")

# Workflow: Process a string with numbers, extract them, and perform calculations
workflow = [
    {
        'task': 'extract_numbers',
        'params': {'text': 'The scores are 85, 92, 78, 95, and 88'}
    },
    {
        'task': 'mean',
        'params': {'use_previous_result': 0, 'result_key': 'numbers'}
    },
    {
        'task': 'median',
        'params': {'use_previous_result': 0, 'result_key': 'numbers'}
    }
]

results = orchestrator.execute_workflow(workflow)

print(f"\nExtracted numbers: {results[0]}")
print(f"Mean score: {results[1]:.2f}")
print(f"Median score: {results[2]:.2f}")

## 9. Example: Another Complex Workflow

Process text and perform mathematical operations on the results.

In [ ]:
print("=== Text Processing + Math Workflow ===")

# Count words in different texts and calculate statistics
texts = [
    "Artificial intelligence is transforming technology",
    "Machine learning enables computers to learn",
    "Deep neural networks power modern AI"
]

word_counts = []
for text in texts:
    count = orchestrator.execute_task('count_words', text=text)
    word_counts.append(count)
    print(f"Text: '{text}'")
    print(f"Word count: {count}\n")

# Calculate average word count
avg_words = orchestrator.execute_task('mean', numbers=word_counts)
print(f"Average word count across all texts: {avg_words:.2f}")

## 10. View Agent History

Check what tasks each agent has performed.

In [ ]:
print("=== Agent Action History ===")

print("\nMath Agent History:")
math_history = orchestrator.get_agent_history('MathAgent')
for i, action in enumerate(math_history[-5:], 1):  # Show last 5 actions
    print(f"{i}. Task: {action['task']}, Result: {action['result']}")

print("\nString Agent History:")
string_history = orchestrator.get_agent_history('StringAgent')
for i, action in enumerate(string_history[-5:], 1):  # Show last 5 actions
    print(f"{i}. Task: {action['task']}, Result: {action['result']}")

## 11. Interactive Section

Try your own tasks with the multi-agent system!

In [ ]:
# Example: Custom tasks
# Uncomment and modify the examples below to try different operations

# Math operations
# result = orchestrator.execute_task('factorial', n=5)
# print(f"5! = {result}")

# String operations
# result = orchestrator.execute_task('replace', text="Hello World", old="World", new="Agent")
# print(f"Result: {result}")

# Your custom workflow
# custom_workflow = [
#     {'task': 'your_task', 'params': {'param1': 'value1'}},
#     # Add more steps...
# ]
# results = orchestrator.execute_workflow(custom_workflow)
# print(results)

## 12. System Information

Display all capabilities of the multi-agent system.

In [ ]:
print("=== Multi-Agent System Capabilities ===")

all_capabilities = orchestrator.get_all_capabilities()

for agent_name, capabilities in all_capabilities.items():
    print(f"\n{agent_name}:")
    print(f"  Total capabilities: {len(capabilities)}")
    print(f"  Available operations: {', '.join(capabilities)}")